Instalar Primero las librerías necesarias

In [ ]:
import re
import random
import unicodedata
import pandas as pd
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords

#obtener stopwords para no confundir al chatbot
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

Por Ahora crear un dataframe que usará la información de catalogo

In [ ]:
#  Catálogo de productos y promociones

import pandas as pd

datos = [
    {
        "producto": "Cookies & Cream",
        "promocion": "2x1",
        "fecha_inicio": "2026-10-01",
        "fecha_fin": "2026-10-31",
        "ejecucion": "Exhibidor de cartón"
    },
    {
        "producto": "Dark",
        "promocion": "20% de descuento",
        "fecha_inicio": "2026-10-05",
        "fecha_fin": "2026-10-25",
        "ejecucion": "Cabecera de góndola"
    },
    {
        "producto": "Milk",
        "promocion": "3x2",
        "fecha_inicio": "2026-10-10",
        "fecha_fin": "2026-11-10",
        "ejecucion": "Display de mostrador"
    }
]

catalogo = pd.DataFrame(datos)

catalogo

,producto,promocion,fecha_inicio,fecha_fin,ejecucion
0,Cookies & Cream,2x1,2026-10-01,2026-10-31,Exhibidor de cartón
1,Dark,20% de descuento,2026-10-05,2026-10-25,Cabecera de góndola
2,Milk,3x2,2026-10-10,2026-11-10,Display de mostrador


Configuración inicial de herramientas NLP

In [ ]:
tokenizer = RegexpTokenizer(r'\w+')
stemmer = SnowballStemmer("spanish")
stop_words = set(stopwords.words("spanish"))
stop_words.discard("cuando")

def normalizar_texto(texto):
    """
    Convierte el texto a minúsculas, elimina acentos y caracteres especiales.
    """
    texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"[^a-zñ0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def tokenizar_y_stem(texto):
    """
    Tokeniza el texto y aplica stemming para reducir variaciones morfológicas.
    """
    texto = normalizar_texto(texto)
    tokens = tokenizer.tokenize(texto)
    tokens = [token for token in tokens if token not in stop_words]
    stems = [stemmer.stem(token) for token in tokens]
    return tokens, stems

Definir intenciones del chatbot

In [ ]:
intenciones = {
    "saludo": {
        "ejemplos": ["hola", "buenas", "buenos dias", "buenas tardes"],
        "respuestas": ["hola cómo te ayudo? dime un producto y qué necesitas saber", "buenos dias, soy tu asistente de ejecución dime de qué producto buscas información",
                       "Hola soy tu asistente de ejecución en que puedo ayudarte?"]
    },
    "despedida":{
        "ejemplos": ["gracias", "adios", "hasta luego", "hasta la proxima"],
        "respuestas" : ["Gracias por ayudarnos a tener una gran ejecución", "Con gusto, siempre a la orden"]
    },
    "consultar_promocion": {
        "ejemplos": ["qué promo tiene","está en promoción","tiene algún descuento","hay oferta"],
        "respuestas": ["la promoción del producto {producto} es {promocion}", "la promoción activa de {producto} es {promocion}"]
    },
    "consultar_ejecucion": {
        "ejemplos": ["debería tener un mueble","debería tener una cabecera","debería haber un display","qué tipo de exhibición debo tener"],
        "respuestas": ["la ejecución del producto {producto} es {ejecucion}", "La correcta ejecución adicional del producto {producto} es {ejecucion}"]
    },
    "consultar_vigencia": {
        "ejemplos": ["cuál es la duración del descuento","qué vigencia tiene","por cuanto tiempo está la promo", "cuando termina", "desde cuando", "acaba", "vigencia"],
        "respuestas": ["la vigencia para el producto {producto} es {vigencia}", "la vigencia del descuento del producto {producto} es {vigencia}"]
    },
}

Preparar el vocabulario por intención

In [ ]:
def construir_vocabulario_intenciones(intenciones):
    """
    Construye una representación basada en stems para cada intención.
    """
    vocabulario = {}

    for nombre_intencion, datos in intenciones.items():
        stems_intencion = []

        for ejemplo in datos["ejemplos"]:
            _, stems = tokenizar_y_stem(ejemplo)
            stems_intencion.extend(stems)

        vocabulario[nombre_intencion] = set(stems_intencion)

    return vocabulario

vocabulario_intenciones = construir_vocabulario_intenciones(intenciones)


In [ ]:
#Probar qué palabras se extraen de alguna intención como test.

vocabulario_intenciones = construir_vocabulario_intenciones(intenciones)

print(vocabulario_intenciones["consultar_promocion"])

{'algun', 'ofert', 'descuent', 'prom', 'promocion'}


Detectar intención

In [ ]:
def detectar_intencion(mensaje):
    """
    Detecta la intención comparando stems del mensaje con el vocabulario
    de cada intención. Devuelve intención y puntuación.
    """
    _, stems_mensaje = tokenizar_y_stem(mensaje)
    stems_mensaje = set(stems_mensaje)

    puntajes = {}

    for intencion, stems_intencion in vocabulario_intenciones.items():
        coincidencias = stems_mensaje.intersection(stems_intencion)
        puntajes[intencion] = len(coincidencias)
        #print(intencion, ":", coincidencias, "->", puntajes[intencion])

    mejor_intencion = max(puntajes, key=puntajes.get)
    mejor_puntaje = puntajes[mejor_intencion]

    if mejor_puntaje == 0:
        return "desconocida", 0

    return mejor_intencion, mejor_puntaje

    columnas_intenciones = {
    "consultar_promocion": ["promocion"],
    "consultar_ejecucion": ["ejecucion"],
    "consultar_vigencia": ["fecha_inicio", "fecha_fin"]
}


In [ ]:
detectar_intencion("acaba la promoción")

('consultar_promocion', 1)

Detección básica de entidades

In [ ]:
catalogo_productos = {
    "Cookies & Cream": ["cookies", "cookies&cream", "cream", "cookies"],
    "Dark": ["dark", "oscuro", "amargo", "dark chocolate", "dark chocolate"],
    "Milk": ["milk", "leche", "chocolate con leche"]

}

def detectar_entidades(mensaje):
    """
    Detecta entidades básicas mediante diccionario de términos.
    """
    texto = normalizar_texto(mensaje)
    entidades = {}

    for entidad, palabras_clave in catalogo_productos.items():
        for palabra in palabras_clave:
            if palabra in texto:
                entidades["producto"] = entidad
                return entidades

    entidades["producto"] = "no especificado"
    return entidades


In [ ]:
print(detectar_entidades("¿Qué promoción tiene el chocolate blanco?"))

{'producto': 'no especificado'}


Generación de respuestas

In [ ]:
columnas_intenciones = {
    "consultar_promocion": ["promocion"],
    "consultar_ejecucion": ["ejecucion"],
    "consultar_vigencia": ["fecha_inicio", "fecha_fin"]
}

def generar_respuesta(mensaje):
    """
    Genera una respuesta considerando intención y entidades.
    """
    intencion, puntaje = detectar_intencion(mensaje)
    entidades = detectar_entidades(mensaje)

    if intencion == "desconocida":
        return {
            "intencion": intencion,
            "puntaje": puntaje,
            "entidades": entidades,
            "respuesta": (
                "No estoy seguro de haber entendido. "
                "Puedes preguntarme por promociones, ejecución o vigencias de productos"
            )
        }

    if intencion  in ["saludo", "despedida"]:
        return {
            "intencion": intencion,
            "puntaje": puntaje,
            "entidades": entidades,
            "respuesta": random.choice(intenciones[intencion]["respuestas"])
        }


    producto = entidades["producto"]
    fila_catalogo = catalogo[catalogo["producto"] == producto]

    if fila_catalogo.empty:
        return {
            "intencion": intencion,
            "puntaje": puntaje,
            "entidades": entidades,
            "respuesta": "No encontré información"
        }


    columnas = columnas_intenciones[intencion]
    valores = fila_catalogo[columnas].iloc[0]

    if intencion == "consultar_vigencia":
      inicio = valores["fecha_inicio"]
      fin = valores["fecha_fin"]

      entidades["vigencia"] = f"desde {inicio} hasta {fin}"
    elif intencion == "consultar_ejecucion":
      entidades["ejecucion"] = valores["ejecucion"]

    elif intencion == "consultar_promocion":
      entidades["promocion"] = valores["promocion"]

    respuesta = random.choice(intenciones[intencion]["respuestas"])
    respuesta = respuesta.format(**entidades)

    return {
        "intencion": intencion,
        "puntaje": puntaje,
        "entidades": entidades,
        "respuesta": respuesta
    }





In [ ]:
resultado = generar_respuesta("¿Qué ejecucion debería tener milk")
print(resultado["respuesta"])

la ejecución del producto Milk es Display de mostrador


In [ ]:
def iniciar_chatbot():
    """
    Ejecuta el chatbot en modo interactivo dentro de Google Colab.
    """
    historial = []

    print("Chatbot: Hola, soy tu asistente virtual de ejecución")
    print("Chatbot: Puedes preguntarme por promociones y ejecución necesaria, solo dime el producto y lo que necesitas saber")
    print("Chatbot: Escribe 'salir' para terminar.\n")

    while True:
        mensaje_usuario = input("Usuario: ")

        if normalizar_texto(mensaje_usuario) == "salir":
            print("Chatbot: Gracias por comunicarte. ¡Hasta pronto!")
            break

        resultado = generar_respuesta(mensaje_usuario)

        historial.append({
            "mensaje_usuario": mensaje_usuario,
            "intencion_detectada": resultado["intencion"],
            "puntaje": resultado["puntaje"],
            "entidades": resultado["entidades"],
            "respuesta": resultado["respuesta"]
        })

        print(f"Chatbot: {resultado['respuesta']}\n")

    return pd.DataFrame(historial)

# Ejecutar chatbot
historial_chat = iniciar_chatbot()
historial_chat


Chatbot: Hola, soy tu asistente virtual de ejecución
Chatbot: Puedes preguntarme por promociones y ejecución necesaria, solo dime el producto y lo que necesitas saber
Chatbot: Escribe 'salir' para terminar.

Usuario: que promo tiene milk
Chatbot: la promoción activa de Milk es 3x2

Usuario: salir
Chatbot: Gracias por comunicarte. ¡Hasta pronto!


,mensaje_usuario,intencion_detectada,puntaje,entidades,respuesta
0,que promo tiene milk,consultar_promocion,1,"{'producto': 'Milk', 'promocion': '3x2'}",la promoción activa de Milk es 3x2


In [ ]:
# ============================================================
# Evaluación básica del chatbot
# ============================================================

casos_prueba = [
    {"mensaje": "hola buenos dias", "intencion_esperada": "saludo"},
    {"mensaje": "gracias hasta luego", "intencion_esperada": "despedida"},

    {"mensaje": "que promocion tiene Dark", "intencion_esperada": "consultar_promocion"},
    {"mensaje": "hay oferta en Milk", "intencion_esperada": "consultar_promocion"},

    {"mensaje": "que tipo de exhibicion debo tener para Dark", "intencion_esperada": "consultar_ejecucion"},
    {"mensaje": "Milk deberia tener una cabecera", "intencion_esperada": "consultar_ejecucion"},

    {"mensaje": "cuando termina la promocion de Dark", "intencion_esperada": "consultar_vigencia"},
    {"mensaje": "que vigencia tiene Milk", "intencion_esperada": "consultar_vigencia"},
    {"mensaje": "por cuanto tiempo esta la promo de Cookies & Cream", "intencion_esperada": "consultar_vigencia"},

    {"mensaje": "quiero reportar una tienda cerrada", "intencion_esperada": "desconocida"}
]

resultados = []

for caso in casos_prueba:
    resultado = generar_respuesta(caso["mensaje"])
    resultados.append({
        "mensaje": caso["mensaje"],
        "intencion_esperada": caso["intencion_esperada"],
        "intencion_detectada": resultado["intencion"],
        "correcto": caso["intencion_esperada"] == resultado["intencion"],
        "respuesta": resultado["respuesta"]
    })

df_resultados = pd.DataFrame(resultados)

precision = df_resultados["correcto"].mean()
tasa_fallback = (df_resultados["intencion_detectada"] == "desconocida").mean()

print("Resultados de evaluación")
print("------------------------")
print(f"Precisión de intención: {precision:.2%}")
print(f"Tasa de fallback: {tasa_fallback:.2%}")

df_resultados

Resultados de evaluación
------------------------
Precisión de intención: 100.00%
Tasa de fallback: 10.00%


,mensaje,intencion_esperada,intencion_detectada,correcto,respuesta
0,hola buenos dias,saludo,saludo,True,hola cómo te ayudo? dime un producto y qué nec...
1,gracias hasta luego,despedida,despedida,True,"Con gusto, siempre a la orden"
2,que promocion tiene Dark,consultar_promocion,consultar_promocion,True,la promoción del producto Dark es 20% de descu...
3,hay oferta en Milk,consultar_promocion,consultar_promocion,True,la promoción activa de Milk es 3x2
4,que tipo de exhibicion debo tener para Dark,consultar_ejecucion,consultar_ejecucion,True,la ejecución del producto Dark es Cabecera de ...
5,Milk deberia tener una cabecera,consultar_ejecucion,consultar_ejecucion,True,la ejecución del producto Milk es Display de m...
6,cuando termina la promocion de Dark,consultar_vigencia,consultar_vigencia,True,la vigencia del descuento del producto Dark es...
7,que vigencia tiene Milk,consultar_vigencia,consultar_vigencia,True,la vigencia para el producto Milk es desde 202...
8,por cuanto tiempo esta la promo de Cookies & C...,consultar_vigencia,consultar_vigencia,True,la vigencia del descuento del producto Cookies...
9,quiero reportar una tienda cerrada,desconocida,desconocida,True,No estoy seguro de haber entendido. Puedes pre...
